Změny oproti původnímu kroku 2 a důvody
1) Ukládání výstupů: Int16 → Float32

Původně: t1_clean i t2_registered se před uložením castovaly na sitkInt16.

Nově: ukládá se sitkFloat32.

Proč: Int16 zbytečně kvantizuje intenzity (zhoršuje metriky podobnosti, vizuální QC a hlavně následný výpočet T1/T2 poměru). Float32 zachová informace po N4 a interpolaci.

2) Uložení transformace registrace (.tfm)

Původně: transformace se neukládala.

Nově: ukládá se sub-XXX_T2toT1.tfm (sitk.WriteTransform).

Proč: audit trail a reprodukovatelnost. Lze kdykoli znovu přesně aplikovat transformaci na originální T2, ověřit registraci nebo přegenerovat T2w_reg bez opětovné optimalizace.

3) Brain maska pro N4 a registraci (preferovaně z infantFS aseg, fallback Otsu)

Původně: pro N4 se používala Otsu maska nad celou hlavou/pozadím (není to brain extraction).

Nově:

pokud existuje infantfs-8.1.0/sub-XXX/mri/aseg.nii.gz, resampluje se do geometrie T1 a vytvoří se maska aseg > 0 (brain mask),

pokud aseg není dostupná, použije se Otsu maska jako fallback.

Proč: maskovaná N4 i registrace jsou stabilnější (metrika není ovlivněna obličejem, krkem, pozadím, ořezem FOV). Odpovídá to kroku “brain extraction” v Soun/Ganzetti, ale robustněji pro neonaty (kde BET bývá problém).

4) N4 bias correction se provádí s maskou (a pro T1 a T2 odděleně)

Původně: N4 se aplikovala na T1 i T2 s Otsu maskou vytvořenou z daného snímku.

Nově:

T1 N4 používá brain masku v prostoru T1 (aseg>0 nebo Otsu),

T2 N4 používá Otsu masku z T2 (protože T2 není ještě zarovnaná na T1).

Proč: N4 má být odhadována v relevantní oblasti (hlava/mozek), jinak může být bias field zkreslený pozadím. Pro T1 máš nejlepší masku z aseg; pro T2 před registrací nemůžeš použít T1 masku (jiná geometrie), proto Otsu na T2.

5) Registrace: single-scale → multi-resolution pyramida

Původně: jedna úroveň, bez pyramidy.

Nově: SetShrinkFactorsPerLevel([4,2,1]) a SetSmoothingSigmasPerLevel([2,1,0]) + SmoothingSigmasAreSpecifiedInPhysicalUnitsOn().

Proč: multi-resolution je standardní stabilizační postup. Nejprve se chytne hrubé zarovnání na downsampleovaných datech a teprve potom se dolaďuje detail. Výrazně snižuje riziko lokálních minim a selhání registrace.

6) Inicializace transformace: GEOMETRY → MOMENTS

Původně: CenteredTransformInitializerFilter.GEOMETRY.

Nově: CenteredTransformInitializerFilter.MOMENTS.

Proč: pro data se šumem, artefakty nebo ořezem FOV bývá inicializace podle momentů intenzity robustnější než čistě geometrický střed.

7) Optimalizér: GradientDescent → RegularStepGradientDescent

Původně: SetOptimizerAsGradientDescent(learningRate=1.0, iterations=100, ...).

Nově: SetOptimizerAsRegularStepGradientDescent(learningRate=2.0, minStep=1e-4, iterations=200, ...).

Proč: RegularStep bývá v praxi stabilnější (kontroluje krok a jeho zmenšování), méně citlivý na volbu learning rate. Zvýšení iterací zlepšuje konvergenci u obtížnějších subjektů.

8) Sampling metriky: 1 % → 5 %

Původně: SetMetricSamplingPercentage(0.01).

Nově: 0.05 (doporučený rozsah 0.02–0.10 podle rychlosti).

Proč: 1 % je u malých struktur (neonatální mozek) často příliš málo a MI se pak odhaduje šumově. 5 % je stále rychlé, ale stabilnější.

9) Registrace je maskovaná (fixed mask)

Původně: metrika byla počítána na celém objemu.

Nově: SetMetricFixedMask(t1_mask) (brain mask v prostoru T1).

Proč: metrika MI se má optimalizovat nad relevantní anatomií (mozek), ne nad pozadím/obličejem. U neonatů se často liší ořez a přítomnost očí apod., což může registraci “tahat” špatným směrem.

10) Interpolátor při resamplingu: BSpline → Linear

Původně: sitkBSpline pro resampling registrované T2.

Nově: sitkLinear.

Proč: BSpline může vytvářet overshoot/ringing a je méně vhodný pro následné kvantitativní operace (dělení T1/T2). Linear je konzervativnější a standardní volba pro intenzitní data před ratio.

11) Přidán výpočet “BEFORE” pro QC metriky

Původně: žádné objektivní metriky se v kroku 2 neukládaly.

Nově: vytváří se t2_before = T2 po N4 jen resamplovaná do T1 (identity transform) a počítá se MI_pos:

MI_before = MI_pos(T1, T2_before)

MI_after = MI_pos(T1, T2_reg)

Proč: umožní objektivní dokumentaci zlepšení registrace (před/po) bez nutnosti čekat na externí QC notebook.

12) Zápis QC do CSV (step2_registration_qc.csv)

Nově: pro každý subjekt se ukládá MI_before, MI_after, ΔMI, final_metric_raw, mask_source.

Proč: jednoduchý tabulkový důkaz pro vedoucího i do práce (kolik subjektů se zlepšilo, outliery). mask_source říká, zda byla použita robustní aseg maska nebo fallback Otsu.

13) Uložení brain masky v prostoru T1

Nově: ukládá se brain_mask_T1.nii.gz do složky pacienta.

Proč: usnadní další kroky (ratio pouze v mozku, další QC, případně výpočet statistik jen v mozku) a zlepší reprodukovatelnost pipeline.

Poznámka k návaznosti na Soun/Ganzetti (do metodiky)

Brain extraction: místo BET/SPM používáš masku z infantFS aseg (robustní pro neonaty).

Bias correction: stejně jako studie provádíš N4 v maskované oblasti.

Registrace: stejně jako studie používáš lineární rigid T2→T1, ale s multirezolucí a maskováním pro stabilitu.

Intenzitní kalibrace eye/muscle: ty ji vědomě vynecháváš (riziko chyb u neonatů) a spoléháš na robustní ROI statistiky (medián)

In [8]:
import os
import csv
import gc
import numpy as np
import SimpleITK as sitk

sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(1)

# =========================
# NASTAVENÍ
# =========================
RESUME_SKIP_DONE = True
SKIP_BY_CSV = True
SKIP_BY_FILES = True

START_FROM_PATIENT = None   # <- změň na None, pokud nechceš resume od konkrétního sub
# START_FROM_PATIENT = None

seznam_souboru = "vybrane_snimky.txt"

vystupni_slozka_hlavni = r"D:\bakalarka\data_hie\derivatives\coregistered2"
PIPELINE_TAG = "v2"
infantfs_dir = r"D:\bakalarka\data_hie\derivatives\infantfs-8.1.0"

qc_csv = os.path.join(vystupni_slozka_hlavni, f"step2_registration_qc_{PIPELINE_TAG}.csv")
os.makedirs(vystupni_slozka_hlavni, exist_ok=True)

# =========================
# NIFTI FIX (qform/sform) pro ITK-SNAP overlay
#   Robustní: nastaví qform/sform podle geometrie SimpleITK (LPS->RAS)
# =========================
try:
    import nibabel as nib
except ImportError as e:
    raise ImportError("Chybí balíček nibabel. Nainstaluj: pip install nibabel") from e

def sitk_to_nifti_affine_ras(img: sitk.Image) -> np.ndarray:
    direction = np.array(img.GetDirection(), dtype=float).reshape(3, 3)
    spacing = np.array(img.GetSpacing(), dtype=float)
    origin = np.array(img.GetOrigin(), dtype=float)

    aff_lps = np.eye(4, dtype=float)
    aff_lps[:3, :3] = direction @ np.diag(spacing)
    aff_lps[:3, 3] = origin

    lps2ras = np.diag([-1, -1, 1, 1]).astype(float)
    return lps2ras @ aff_lps

def fix_nifti_qform_sform_from_sitk(nifti_path: str, sitk_img: sitk.Image):
    img = nib.load(nifti_path)
    aff = sitk_to_nifti_affine_ras(sitk_img)

    hdr = img.header.copy()
    hdr.set_qform(aff, code=1)
    hdr.set_sform(aff, code=1)

    data = np.asanyarray(img.dataobj)
    nib.save(nib.Nifti1Image(data, aff, header=hdr), nifti_path)

# =========================
# HELPERY
# =========================
def otsu_mask(img: sitk.Image) -> sitk.Image:
    img_f = sitk.Cast(img, sitk.sitkFloat32)
    m = sitk.OtsuThreshold(img_f, 0, 1, 200)
    return sitk.Cast(m, sitk.sitkUInt8)

def try_brain_mask_from_aseg(t1_img: sitk.Image, aseg_path: str):
    if not os.path.exists(aseg_path):
        return None
    aseg = sitk.ReadImage(aseg_path)
    aseg_rs = sitk.Resample(
        aseg, t1_img, sitk.Transform(),
        sitk.sitkNearestNeighbor, 0.0, aseg.GetPixelID()
    )
    return sitk.Cast(aseg_rs > 0, sitk.sitkUInt8)

def n4_bias_correct(img: sitk.Image, mask: sitk.Image, shrink: int = 2) -> sitk.Image:
    """
    Kompatibilní N4 pro SimpleITK bez SetShrinkFactor():
    - zmenší img+mask (shrink)
    - provede N4 na zmenšeném objemu
    - získá log-bias field, přeresampluje na plné rozlišení
    - aplikuje korekci na plný obraz
    """
    img_f = sitk.Cast(img, sitk.sitkFloat32)
    mask_u8 = sitk.Cast(mask, sitk.sitkUInt8)

    def _shrink(im: sitk.Image, sh: int) -> sitk.Image:
        if sh <= 1:
            return im
        return sitk.Shrink(im, [sh, sh, sh])

    def _run(sh: int) -> sitk.Image:
        img_s = _shrink(img_f, sh)
        mask_s = _shrink(mask_u8, sh)

        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([50, 50, 30, 20])
        corrector.SetConvergenceThreshold(1e-6)

        # N4 na zmenšeném
        _ = corrector.Execute(img_s, mask_s)

        # preferovaný způsob: log bias field -> resample -> korekce na full-res
        if hasattr(corrector, "GetLogBiasFieldAsImage"):
            log_bias_s = corrector.GetLogBiasFieldAsImage(img_s)
            log_bias_full = sitk.Resample(
                sitk.Cast(log_bias_s, sitk.sitkFloat32),
                img_f,
                sitk.Transform(),
                sitk.sitkBSpline,
                0.0,
                sitk.sitkFloat32
            )
            corrected = img_f / sitk.Exp(log_bias_full)
            return sitk.Cast(corrected, sitk.sitkFloat32)

        # fallback: resample corrected image (méně ideální, ale funkční)
        corrected_s = corrector.Execute(img_s, mask_s)
        corrected_full = sitk.Resample(
            sitk.Cast(corrected_s, sitk.sitkFloat32),
            img_f,
            sitk.Transform(),
            sitk.sitkLinear,
            0.0,
            sitk.sitkFloat32
        )
        return corrected_full

    try:
        return _run(shrink)
    except RuntimeError as e:
        msg = str(e)
        if ("Failed to allocate memory" in msg) or ("allocate memory" in msg):
            if shrink < 4:
                print(f"    [N4] Nedostatek paměti, retry shrink={shrink*2}")
                return _run(shrink * 2)
            if shrink < 8:
                print("    [N4] Nedostatek paměti, retry shrink=8")
                return _run(8)
        raise
    finally:
        del img_f, mask_u8
        gc.collect()

def MI_pos(fixed: sitk.Image, moving: sitk.Image, fixed_mask=None) -> float:
    fixed_f = sitk.Cast(fixed, sitk.sitkFloat32)
    moving_f = sitk.Cast(moving, sitk.sitkFloat32)

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    if fixed_mask is not None:
        reg.SetMetricFixedMask(sitk.Cast(fixed_mask, sitk.sitkUInt8))

    mi_raw = reg.MetricEvaluate(fixed_f, moving_f)
    return float(-mi_raw)

def _physical_center(img: sitk.Image):
    size = img.GetSize()
    idx = [(s - 1) / 2.0 for s in size]
    return img.TransformContinuousIndexToPhysicalPoint(idx)

def rigid_register_pre_to_t1(fixed: sitk.Image, moving_pre: sitk.Image, fixed_mask=None):
    fixed = sitk.Cast(fixed, sitk.sitkFloat32)
    moving = sitk.Cast(moving_pre, sitk.sitkFloat32)

    init = sitk.Euler3DTransform()
    init.SetCenter(_physical_center(fixed))

    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)

    if fixed_mask is not None:
        if sitk.GetArrayViewFromImage(fixed_mask).sum() > 0:
            reg.SetMetricFixedMask(sitk.Cast(fixed_mask, sitk.sitkUInt8))

    reg.SetMetricSamplingStrategy(reg.REGULAR)
    reg.SetMetricSamplingPercentage(0.20)
    if hasattr(reg, "SetMetricSamplingPercentagePerLevel"):
        reg.SetMetricSamplingPercentagePerLevel([0.30, 0.20, 0.10])

    reg.SetInterpolator(sitk.sitkLinear)

    reg.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.0,
        minStep=1e-4,
        numberOfIterations=250,
        gradientMagnitudeTolerance=1e-8
    )
    reg.SetOptimizerScalesFromPhysicalShift()

    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    reg.SetInitialTransform(init, inPlace=False)

    final_tfm = reg.Execute(fixed, moving)
    final_metric = reg.GetMetricValue()

    moving_reg = sitk.Resample(
        moving, fixed, final_tfm,
        sitk.sitkLinear, 0.0, sitk.sitkFloat32
    )
    return moving_reg, final_tfm, float(final_metric)

def load_done_patients_from_csv(qc_csv_path: str) -> set[str]:
    done = set()
    if not os.path.exists(qc_csv_path):
        return done
    with open(qc_csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames or "patient" not in reader.fieldnames:
            return done
        for row in reader:
            p = (row.get("patient") or "").strip()
            status = (row.get("status") or "").strip()
            tag = (row.get("tag") or "").strip()
            if p and (not tag or tag == PIPELINE_TAG) and status == "OK":
                done.add(p)
    return done

# =========================
# HLAVNÍ BĚH
# =========================
if not os.path.exists(seznam_souboru):
    raise FileNotFoundError(f"CHYBA: Soubor {seznam_souboru} neexistuje.")

with open(seznam_souboru, "r", encoding="utf-8") as f:
    radky = [r.strip() for r in f.readlines() if r.strip()]

print(f"Načteno {len(radky)} pacientů. Spouštím krok 2... (tag={PIPELINE_TAG})")
print("Výstupní složka:", vystupni_slozka_hlavni)
print("QC CSV:", qc_csv)

# RESUME z CSV (načti před otevřením append writeru)
done_patients = set()
if RESUME_SKIP_DONE and SKIP_BY_CSV:
    done_patients = load_done_patients_from_csv(qc_csv)
    print(f"[RESUME] V QC CSV už je hotových: {len(done_patients)} pacientů")

# připrav QC CSV (append-safe)
write_header = not os.path.exists(qc_csv)
qc_f = open(qc_csv, "a", newline="", encoding="utf-8")
qc_w = csv.writer(qc_f)
if write_header:
    qc_w.writerow(["patient", "MI_before", "MI_after", "dMI", "final_metric_raw",
                   "mask_source", "tag", "status"])

# resume od konkrétního pacienta (volitelné)
started = (START_FROM_PATIENT is None)

for idx, radek in enumerate(radky, start=1):
    pacient, t1_cesta, t2_cesta = radek.split(",", 2)
    print(f"\n[{idx}/{len(radky)}] {pacient}")

    # --- START_FROM_PATIENT ---
    if not started:
        if pacient != START_FROM_PATIENT:
            print(f"  [SKIP] čekám na {START_FROM_PATIENT}")
            continue
        started = True
        print(f"  [RESUME] pokračuji od {START_FROM_PATIENT}")

    # cesty výstupů (definuj jednou a používej všude)
    pacient_out_dir = os.path.join(vystupni_slozka_hlavni, pacient)
    os.makedirs(pacient_out_dir, exist_ok=True)

    t1_out    = os.path.join(pacient_out_dir, f"{pacient}_T1w_N4_{PIPELINE_TAG}.nii.gz")
    t2pre_out = os.path.join(pacient_out_dir, f"{pacient}_T2w_pre_{PIPELINE_TAG}.nii.gz")
    t2_out    = os.path.join(pacient_out_dir, f"{pacient}_T2w_reg_{PIPELINE_TAG}.nii.gz")
    tfm_out   = os.path.join(pacient_out_dir, f"{pacient}_T2pre_toT1_{PIPELINE_TAG}.tfm")
    mask_out  = os.path.join(pacient_out_dir, f"brain_mask_T1_{PIPELINE_TAG}.nii.gz")

    # --- RESUME: CSV ---
    if RESUME_SKIP_DONE and SKIP_BY_CSV and pacient in done_patients:
        print("  [SKIP] už hotovo (QC CSV)")
        continue

    # --- RESUME: soubory ---
    if RESUME_SKIP_DONE and SKIP_BY_FILES and all(os.path.exists(p) for p in [t1_out, t2pre_out, t2_out, tfm_out, mask_out]):
        print("  [SKIP] už hotovo (výstupy existují)")
        continue

    # Load
    t1_img = sitk.ReadImage(t1_cesta)
    t2_img = sitk.ReadImage(t2_cesta)

    # T1 maska (aseg -> fallback Otsu)
    aseg_path = os.path.join(infantfs_dir, pacient, "mri", "aseg.nii.gz")
    t1_mask = try_brain_mask_from_aseg(t1_img, aseg_path)
    mask_source = "aseg>0" if t1_mask is not None else "otsu"
    if t1_mask is None:
        t1_mask = otsu_mask(t1_img)

    # N4
    print("  -> N4 bias correction (T1 + T2)...")
    try:
        t1_n4 = n4_bias_correct(t1_img, t1_mask)
    except RuntimeError as e:
        print(f"  [FAIL] N4(T1) selhalo: {e}")
        qc_w.writerow([pacient, "nan", "nan", "nan", "nan", mask_source, PIPELINE_TAG, "N4_FAIL"])
        continue

    t2_mask = otsu_mask(t2_img)
    try:
        t2_n4 = n4_bias_correct(t2_img, t2_mask)
    except RuntimeError as e:
        print(f"  [FAIL] N4(T2) selhalo: {e}")
        qc_w.writerow([pacient, "nan", "nan", "nan", "nan", mask_source, PIPELINE_TAG, "N4_FAIL"])
        continue

    # PREALIGN: T2 do geometrie T1
    print("  -> Prealign: resample T2 do geometrie T1 (identity)...")
    t2_pre = sitk.Resample(t2_n4, t1_n4, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32)

    # Registrace (fine-tune)
    print("  -> Rigid registrace (fine-tune) na T2_pre ...")
    try:
        t2_reg, tfm, final_metric_raw = rigid_register_pre_to_t1(t1_n4, t2_pre, fixed_mask=t1_mask)
    except RuntimeError as e:
        print(f"  [FAIL] Registrace selhala: {e}")
        qc_w.writerow([pacient, "nan", "nan", "nan", "nan", mask_source, PIPELINE_TAG, "REG_FAIL"])
        # cleanup
        del t1_img, t2_img, t1_n4, t2_n4, t2_pre
        gc.collect()
        continue

    # QC metriky
    mi_b = MI_pos(t1_n4, t2_pre, fixed_mask=t1_mask)
    mi_a = MI_pos(t1_n4, t2_reg, fixed_mask=t1_mask)
    dmi = mi_a - mi_b
    print(f"  -> MI_pos: {mi_b:.3f} → {mi_a:.3f} (Δ {dmi:.3f}) | mask={mask_source}")

    qc_w.writerow([pacient, f"{mi_b:.6f}", f"{mi_a:.6f}", f"{dmi:.6f}",
                   f"{final_metric_raw:.6f}", mask_source, PIPELINE_TAG, "OK"])

    # Ukládání
    print("  -> Ukládám výstupy (a fixuju qform/sform pro ITK-SNAP)...")
    sitk.WriteImage(sitk.Cast(t1_n4, sitk.sitkFloat32), t1_out)
    sitk.WriteImage(sitk.Cast(t2_pre, sitk.sitkFloat32), t2pre_out)
    sitk.WriteImage(sitk.Cast(t2_reg, sitk.sitkFloat32), t2_out)
    sitk.WriteTransform(tfm, tfm_out)
    sitk.WriteImage(sitk.Cast(t1_mask, sitk.sitkUInt8), mask_out)

    # Fix qform/sform podle SITK geometrie (nejstabilnější pro ITK-SNAP overlay)
    fix_nifti_qform_sform_from_sitk(t1_out, t1_n4)
    fix_nifti_qform_sform_from_sitk(t2pre_out, t2_pre)
    fix_nifti_qform_sform_from_sitk(t2_out, t2_reg)

    # update done set (aby v rámci stejného běhu šel skip)
    done_patients.add(pacient)

    print("  [HOTOVO]")
    print("     ", os.path.basename(t1_out))
    print("     ", os.path.basename(t2pre_out))
    print("     ", os.path.basename(t2_out))
    print("     ", os.path.basename(tfm_out))

    # cleanup per patient
    del t1_img, t2_img, t1_n4, t2_n4, t2_pre, t2_reg, tfm, t1_mask, t2_mask
    gc.collect()

qc_f.close()
print(f"\nVŠECHNO HOTOVO (KROK 2). QC metriky: {qc_csv}")

Načteno 45 pacientů. Spouštím krok 2... (tag=v2)
Výstupní složka: D:\bakalarka\data_hie\derivatives\coregistered2
QC CSV: D:\bakalarka\data_hie\derivatives\coregistered2\step2_registration_qc_v2.csv
[RESUME] V QC CSV už je hotových: 0 pacientů

[1/45] sub-002
  [SKIP] už hotovo (výstupy existují)

[2/45] sub-003
  [SKIP] už hotovo (výstupy existují)

[3/45] sub-004
  [SKIP] už hotovo (výstupy existují)

[4/45] sub-005
  [SKIP] už hotovo (výstupy existují)

[5/45] sub-006
  [SKIP] už hotovo (výstupy existují)

[6/45] sub-007
  [SKIP] už hotovo (výstupy existují)

[7/45] sub-008
  [SKIP] už hotovo (výstupy existují)

[8/45] sub-009
  [SKIP] už hotovo (výstupy existují)

[9/45] sub-011
  [SKIP] už hotovo (výstupy existují)

[10/45] sub-012
  [SKIP] už hotovo (výstupy existují)

[11/45] sub-013
  [SKIP] už hotovo (výstupy existují)

[12/45] sub-014
  [SKIP] už hotovo (výstupy existují)

[13/45] sub-015
  -> N4 bias correction (T1 + T2)...
    [N4] Nedostatek paměti, retry shrink=4
  [FAIL]